# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/) and is accessible at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

We will reference all dataset elements (record sets, fields, columns) by their `@id` values, as this is the canonical way defined by the Croissant specification.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset and its metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
# dataset.metadata is a CroissantMetadata object
metadata_dict = json.loads(dataset.metadata.to_json())  # For pretty printing below only

print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review all available record sets (tables), with their associated `@id` values and fields.

> **Note:** As per the Croissant schema, all data entities (record sets, fields, etc.) are referenced by `@id`.

We'll print a summary of record sets and their columns. These `@id` values are needed for extraction and EDA steps downstream.

In [ ]:
# List record sets
record_sets = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record set: {rs.name}\n  @id: {rs.id}\n  Columns:")
    for col in rs.columns:
        print(f"    - {col.name} (@id: {col.id}, type: {col.data_type})")
    print()

For demonstration, let's look at a sample record from the **first record set** using its `@id`. We will also display sample keys of records for transparency.

In [ ]:
# For this dataset, let's use the first record set found
if not record_sets:
    raise RuntimeError('No record sets found in the dataset metadata.')

rs_main = record_sets[0]  # You can change to a specific record set if needed
main_record_set_id = rs_main.id

sample_record = next(dataset.records(record_set=main_record_set_id))
print(f"First record from record set {main_record_set_id}:")
print(json.dumps(sample_record, indent=2))
print(f"\nAvailable keys in this record: {list(sample_record.keys())}")

## 3. Data Extraction
Load all records from each record set into Pandas DataFrames using the appropriate `@id`.

> For demonstration, we'll load and display the first five records from the main record set.

In [ ]:
dfs = {}
record_set_ids = [rs.id for rs in record_sets]

for rs_id in record_set_ids:
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    dfs[rs_id] = df
    print(f"Loaded record set: {rs_id} with shape {df.shape}")

# Check columns in the main record set DataFrame
print(f"\nColumns in primary record set ({main_record_set_id}):\n{dfs[main_record_set_id].columns.tolist()}")
dfs[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical analysis steps using columns referenced by their `@id` as obtained above.

We'll demonstrate filtering, normalization, and grouping for a chosen numeric field and a group-by field. Adjust the field `@id` variables below to your needs based on available columns.

In [ ]:
# Example: suppose '@id' of a numeric column is 'age_at_second_crc_diagnosis' and group field is 'sex'.
# Please change these variables according to your schema (see print outputs above for real @ids).
# For the purposes of the FAIR^2 dataset, sample possible field ids could be:
#   - age_at_second_crc_diagnosis
#   - sex
# If unsure, examine dfs[main_record_set_id].columns.tolist()

# Update these to match actual column @id in your dataset (these are plausible guesses):
numeric_field_id = 'age_at_second_crc_diagnosis'  # Example name, adjust as needed
group_field_id = 'sex'  # Example name, adjust as needed

main_df = dfs[main_record_set_id].copy()

if numeric_field_id in main_df.columns:
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    num_na = main_df[numeric_field_id].isna().sum()
    print(f"Number of NA in {numeric_field_id}: {num_na}")
    
    threshold = 40  # Example age threshold for filter
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"\nRecords with {numeric_field_id} > {threshold}:")
    display(filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized values of {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized", group_field_id]].head())
    
    # Group by
    if group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'std', 'count'])
        print(f"\n{numeric_field_id} statistics by {group_field_id}:")
        display(grouped)
else:
    print(f"Field '{numeric_field_id}' not found in record set '{main_record_set_id}'. Please review column names above.")

## 5. Visualization
Visualize distributions or relationships between fields using matplotlib or seaborn.

Below is a histogram and a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if field is present
if numeric_field_id in main_df.columns and group_field_id in main_df.columns:
    plt.figure(figsize=(10, 4))
    sns.histplot(data=main_df, x=numeric_field_id, hue=group_field_id, kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² dataset using `mlcroissant`, referencing all entities by their `@id` as per the Croissant standard. We reviewed available record sets and fields, extracted records to dataframes, and performed some basic filtering, normalization, and visualization.

**Key observations:**
- The dataset contains rich clinicopathological variables for second primary colorectal cancer in cancer survivors.
- No missing records were reported (according to the provided metadata).
- Using Croissant's `@id` referencing enables robust and reproducible data workflows.

For more advanced analyses, refer to `mlcroissant` [documentation](https://github.com/mlcommons/croissant) and further domain-specific EDA.
